# Notebook 01 — Construcción y preprocesamiento del dataset facial

**Curso:** Deep Learning – Conceptos · Especialización en Inteligencia Artificial · Universidad de Cundinamarca  
**Actividad:** R2-A1-S10 Redes Neuronales V (semana 10: reconocimiento de rostros)  
**Autor:** Juan Pablo Cristancho Tula

En este primer notebook armo el **dataset de prueba** con fotografías mías y de una persona adulta que me autorizó
por escrito a usar sus fotos con fines académicos. Antes de que una red siamesa pueda comparar rostros, cada foto
tiene que pasar por un preprocesamiento que deje todas las caras "en igualdad de condiciones". El recorrido es este:

1. Cargo las imágenes tal como salen del celular (JPG, PNG, **HEIC** del iPhone y **DNG** en formato RAW) y corrijo su
   orientación.
2. Detecto el rostro y cinco puntos de referencia de la cara con **MTCNN**.
3. **Alineo** la cara girando la imagen hasta que los ojos queden a la misma altura.
4. **Recorto** un cuadrado alrededor del rostro y lo llevo a 160 × 160 píxeles.
5. Reviso la **calidad** de cada recorte (nitidez, brillo y tamaño del rostro) y dejo registro de todo.

Este mismo preprocesamiento lo aplico después, en el Notebook 02, a las imágenes con las que entreno el modelo
(LFW). Así la red siempre ve las caras con el mismo encuadre, tanto al aprender como al probarla.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Juanpatula2361/Taller-Redes-Neuronales-V/blob/main/notebooks/01_preprocesamiento_dataset_facial.ipynb)

## Consentimiento y cuidado de los datos

Trabajar con rostros implica manejar datos personales, así que fijé unas reglas desde el inicio:

- Solo uso fotografías mías y de **personas adultas** que firmaron la autorización de uso de imagen
  (`docs/autorizacion_uso_imagen.md`). No incluyo menores de edad.
- Varias fotos eran de reuniones familiares. En esos casos recorté la imagen para dejar únicamente a la persona
  autorizada y difuminé a cualquier otra persona que quedara cerca.
- Las fotografías **no se suben a GitHub**: la carpeta del dataset está excluida en el `.gitignore` y la mantengo en
  Google Drive. En el repositorio solo queda el código y las figuras de resultados.
- En la demostración del Notebook 03, los rostros que el sistema no reconoce se **difuminan** automáticamente.

In [ ]:
# ============================================================
# 0. Configuración (único lugar donde se cambian rutas y parámetros)
# ============================================================
import os
CONFIG = {
    "USAR_DRIVE": True,
    "CARPETA_BASE": "/content/drive/MyDrive/Redes_Neuronales_V",
    "REPO_URL": "https://github.com/Juanpatula2361/Taller-Redes-Neuronales-V.git",
    "TAM": 160,                    # lado del recorte facial (px)
    "MARGEN": 0.25,                # margen alrededor del rostro detectado
    "CONFIANZA_MIN": 0.90,         # confianza mínima de MTCNN
    "MIN_FOTOS_POR_PERSONA": 5,
    "EXCLUIR_POR_CALIDAD": True,   # descarta imágenes borrosas o con rostro muy pequeño
}
BASE = CONFIG["CARPETA_BASE"]
RAW = os.path.join(BASE, "dataset_raw")             # dataset_raw/<persona>/*.jpg|heic|dng
PROC = os.path.join(BASE, "dataset_procesado")      # salida: dataset_procesado/<persona>/*.png
DESC = os.path.join(BASE, "dataset_descartado")
RES = os.path.join(BASE, "results")
CARPETAS = [RAW, PROC, DESC, RES]

In [ ]:
# ============================================================
# Configuración del entorno: Drive, dependencias y módulo de preprocesamiento
# ============================================================
import sys, json, time, subprocess, warnings
warnings.filterwarnings("ignore")
EN_COLAB = "google.colab" in sys.modules

if EN_COLAB and CONFIG["USAR_DRIVE"]:
    from google.colab import drive
    drive.mount("/content/drive")

if EN_COLAB:
    # MTCNN (detector facial) y lectores de HEIC (iPhone) y DNG (RAW)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mtcnn==1.0.0", "pillow-heif", "rawpy"], check=True)
    if not os.path.isdir("/content/Taller-Redes-Neuronales-V"):
        subprocess.run(["git", "clone", "-q", CONFIG["REPO_URL"], "/content/Taller-Redes-Neuronales-V"])

# El código de preprocesamiento está en scripts/preprocesamiento_facial.py (un solo origen para los 3 notebooks)
for ruta in ["/content/Taller-Redes-Neuronales-V/scripts", os.path.join(BASE, "scripts"),
             os.path.abspath("../scripts"), os.path.abspath("scripts")]:
    if os.path.isfile(os.path.join(ruta, "preprocesamiento_facial.py")):
        sys.path.insert(0, ruta)
        break
else:
    raise FileNotFoundError("No se encontró scripts/preprocesamiento_facial.py: suba el repositorio a GitHub "
                            "o copie la carpeta scripts/ dentro de CARPETA_BASE.")
import preprocesamiento_facial as pf

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
for carpeta in CARPETAS:
    os.makedirs(carpeta, exist_ok=True)
print("Colab:", EN_COLAB, "| carpeta base:", BASE)

## 1. Organización del dataset

Organicé las fotos con una carpeta por persona; el nombre de la carpeta funciona como la etiqueta de identidad:

```
Redes_Neuronales_V/
├── dataset_raw/
│   ├── persona_01/   fotos variadas: distintos días, luz, ángulos, gafas, gorra
│   └── persona_02/
├── dataset_procesado/   (lo genera este notebook)
└── results/             (figuras y tablas para el informe)
```

Algo que aprendí armando el dataset es que importa más la **variedad dentro de cada persona** que la cantidad: fotos
de días distintos, con luz natural y artificial, de frente y de lado, con y sin gafas. También es clave que en cada
foto haya **un solo rostro principal**, para que no haya duda de a quién corresponde la etiqueta.

In [ ]:
# ============================================================
# 1. Inventario del dataset
# ============================================================
personas = sorted(d for d in os.listdir(RAW) if os.path.isdir(os.path.join(RAW, d)) and not d.startswith("."))
assert personas, f"No hay carpetas de personas en {RAW}"
filas = []
for p in personas:
    rutas = pf.listar_imagenes(os.path.join(RAW, p))
    formatos = pd.Series([os.path.splitext(r)[1].lower() for r in rutas]).value_counts().to_dict()
    filas.append({"persona": p, "imagenes": len(rutas), "formatos": formatos})
inventario = pd.DataFrame(filas)
display(inventario)
for _, f in inventario.iterrows():
    if f["imagenes"] < CONFIG["MIN_FOTOS_POR_PERSONA"]:
        print(f"Aviso: {f['persona']} tiene {f['imagenes']} fotos; se recomiendan al menos {CONFIG['MIN_FOTOS_POR_PERSONA']}.")
print("Total de imágenes:", inventario["imagenes"].sum())

## 2. Carga y corrección de orientación

Las fotos del celular guardan la orientación en sus metadatos (EXIF) en lugar de girar los píxeles. Si no se corrige,
muchas imágenes llegan acostadas y el detector simplemente no encuentra la cara. La función `pf.cargar_imagen` aplica
`ImageOps.exif_transpose`, convierte los archivos HEIC y DNG a RGB y limita el lado mayor a 1.600 píxeles para que la
detección sea más rápida.

In [ ]:
# ============================================================
# 2. Vista previa de las imágenes originales (miniaturas)
# ============================================================
muestras = [(p, r) for p in personas for r in pf.listar_imagenes(os.path.join(RAW, p))][:12]
cols = 6
fig, axes = plt.subplots(int(np.ceil(len(muestras) / cols)), cols, figsize=(14, 2.6 * int(np.ceil(len(muestras) / cols))))
for ax in np.ravel(axes):
    ax.axis("off")
for ax, (p, r) in zip(np.ravel(axes), muestras):
    img = pf.cargar_imagen(r, lado_max=600)
    ax.imshow(img)
    ax.set_title(f"{p}\n{os.path.basename(r)[:18]}", fontsize=8)
plt.suptitle("Imágenes originales (orientación corregida)")
plt.tight_layout(); plt.savefig(f"{RES}/01_imagenes_originales.png", dpi=120); plt.show()

## 3. Detección facial con MTCNN

Para encontrar la cara uso **MTCNN** (Zhang et al., 2016), que en realidad son tres redes convolucionales en cascada:
la primera (**P-Net**) propone muchas regiones candidatas, la segunda (**R-Net**) descarta las que no son rostros y la
tercera (**O-Net**) entrega la caja final, su nivel de confianza y **cinco puntos faciales**: los dos ojos, la nariz y
las comisuras de la boca. Los puntos de los ojos son los que necesito en el paso siguiente para alinear.

Hubo dos casos en los que el detector falló al primer intento. Uno fue por las gafas deportivas y la capucha, y lo
resolví repitiendo la búsqueda con la imagen reducida o ampliada. El otro fue una foto con luz verde intensa: ahí busco
la cara sobre una copia en escala de grises, pero el recorte lo hago sobre la imagen a color.

In [ ]:
# ============================================================
# 3. Detección de rostros y puntos faciales
# ============================================================
detector = pf.crear_detector()
ruta_ej = pf.listar_imagenes(os.path.join(RAW, personas[0]))[0]
img_ej = pf.cargar_imagen(ruta_ej)
caras = pf.detectar_rostros(img_ej, detector, confianza_min=CONFIG["CONFIANZA_MIN"])
print(f"{os.path.basename(ruta_ej)}: {len(caras)} rostro(s) detectado(s)")

vis = img_ej.copy()
for c in caras:
    x, y, w, h = c["box"]
    cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 0), max(2, w // 60))
    for nombre, (px, py) in c["keypoints"].items():
        cv2.circle(vis, (int(px), int(py)), max(3, w // 40), (255, 0, 0), -1)
    cv2.putText(vis, f"{c['confidence']:.3f}", (x, max(0, y - 10)), cv2.FONT_HERSHEY_SIMPLEX, w / 250, (0, 255, 0), 2)
plt.figure(figsize=(6, 7)); plt.imshow(vis); plt.axis("off")
plt.title("MTCNN: caja del rostro, confianza y 5 puntos faciales")
plt.tight_layout(); plt.savefig(f"{RES}/01_deteccion_mtcnn.png", dpi=130); plt.show()

## 4. Alineación y recorte

Con los centros de los ojos $(x_i, y_i)$ y $(x_d, y_d)$ calculo qué tan inclinada está la cara:
$\theta = \operatorname{atan2}(y_d - y_i,\; x_d - x_i)$. Luego giro la imagen $\theta$ grados alrededor del punto medio
entre los ojos, recorto un cuadrado de lado $\max(w,h)\,(1 + 2m)$ centrado en el rostro, con un margen $m = 0.25$, y
lo redimensiono a $160 \times 160$ píxeles.

El resultado es que en todas las imágenes los ojos quedan a la misma altura y la cara ocupa más o menos la misma
proporción del recorte. Eso le quita a la red un trabajo innecesario: no tiene que aprender que una cara inclinada
sigue siendo la misma cara.

In [ ]:
# ============================================================
# 4. Antes y después de la alineación
# ============================================================
import inspect
print(inspect.getsource(pf.alinear_y_recortar))

cara = pf.seleccionar_rostro(caras, img_ej.shape, "mayor")
x, y, w, h = cara["box"]
lado = int(max(w, h) * 1.5)
cx, cy = x + w // 2, y + h // 2
sin_alinear = img_ej[max(0, cy - lado // 2): cy + lado // 2, max(0, cx - lado // 2): cx + lado // 2]
alineado, angulo = pf.alinear_y_recortar(img_ej, cara, tam=CONFIG["TAM"], margen=CONFIG["MARGEN"])

fig, axes = plt.subplots(1, 2, figsize=(7, 3.8))
axes[0].imshow(sin_alinear); axes[0].set_title("Recorte sin alinear")
axes[1].imshow(alineado); axes[1].set_title(f"Alineado ({angulo:+.1f}°) y 160×160")
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.savefig(f"{RES}/01_alineacion.png", dpi=130); plt.show()

## 5. Procesamiento de todo el dataset y control de calidad

Ahora aplico el proceso completo a cada foto y guardo un registro con el número de rostros encontrados, la confianza
de la detección, el ángulo que se corrigió y la escala a la que se detectó la cara. Además mido tres indicadores de
calidad sobre el recorte:

- **Nitidez**: la varianza del laplaciano. Si es muy baja, la foto está desenfocada o movida.
- **Brillo** y **contraste**: el promedio y la desviación estándar de la luminancia.

Las imágenes borrosas o con una cara demasiado pequeña las aparto en `dataset_descartado/`, para no contaminar las
pruebas con fotos que ni una persona reconocería con facilidad.

In [ ]:
# ============================================================
# 5. Preprocesamiento completo con registro
# ============================================================
registro = []
t0 = time.perf_counter()
for p in personas:
    os.makedirs(os.path.join(PROC, p), exist_ok=True)
    for r in pf.listar_imagenes(os.path.join(RAW, p)):
        img = pf.cargar_imagen(r)
        res = pf.procesar_imagen(img, detector, tam=CONFIG["TAM"], margen=CONFIG["MARGEN"],
                                 confianza_min=CONFIG["CONFIANZA_MIN"])
        fila = {"persona": p, "archivo": os.path.basename(r), "formato": os.path.splitext(r)[1].lower(),
                "resolucion": f"{img.shape[1]}x{img.shape[0]}"}
        if res is None:
            fila.update({"rostros": 0, "aceptada": False, "avisos": "sin rostro"})
            registro.append(fila)
            continue
        aceptada = not (CONFIG["EXCLUIR_POR_CALIDAD"] and ({"borrosa", "rostro pequeño"} & set(res.avisos)))
        destino = os.path.join(PROC if aceptada else DESC, p, os.path.splitext(os.path.basename(r))[0] + ".png")
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        Image.fromarray(res.recorte).save(destino)
        fila.update({"rostros": res.n_rostros, "confianza": round(res.confianza, 4), "angulo": round(res.angulo, 1),
                     "escala": res.escala, "nitidez": round(res.calidad["nitidez"], 1),
                     "brillo": round(res.calidad["brillo"], 1), "contraste": round(res.calidad["contraste"], 1),
                     "aceptada": aceptada, "avisos": ", ".join(res.avisos) or "-"})
        if res.n_rostros > 1:
            fila["avisos"] = (fila["avisos"] + "; varios rostros: se tomó el mayor").strip("-; ")
        registro.append(fila)
registro = pd.DataFrame(registro)
registro.to_csv(f"{RES}/01_registro_preprocesamiento.csv", index=False)
print(f"Tiempo de preprocesamiento: {time.perf_counter() - t0:.1f} s")
display(registro)
print(registro.groupby("persona")["aceptada"].agg(["count", "sum"]).rename(columns={"count": "imagenes", "sum": "aceptadas"}))

In [ ]:
# ============================================================
# 5b. Rostros procesados por persona (entrada de la red siamesa)
# ============================================================
X_prop, y_prop, nombres_prop, rutas_prop = pf.cargar_dataset_procesado(PROC)
max_cols = max(np.bincount(y_prop))
fig, axes = plt.subplots(len(nombres_prop), max_cols, figsize=(1.6 * max_cols, 1.9 * len(nombres_prop)), squeeze=False)
for ax in axes.ravel():
    ax.axis("off")
for i, nombre in enumerate(nombres_prop):
    for j, k in enumerate(np.flatnonzero(y_prop == i)):
        axes[i, j].imshow(X_prop[k]); axes[i, j].set_title(os.path.basename(rutas_prop[k])[:14], fontsize=7)
    axes[i, 0].text(-20, 80, nombre, fontsize=9, ha="right", va="center")
plt.suptitle(f"Dataset procesado: {len(X_prop)} rostros alineados de 160×160 px")
plt.tight_layout(); plt.savefig(f"{RES}/01_rostros_procesados.png", dpi=130); plt.show()

## 6. Normalización de iluminación (análisis)

Probé la ecualización adaptativa **CLAHE** sobre la luminancia, que ayuda bastante con sombras y contraluz. Sin
embargo, decidí mostrarla solo como análisis y **no aplicarla** al dataset. La razón es que el modelo se entrena con
rostros sin ecualizar; si transformo solo las fotos de prueba, estaría creando una diferencia entre lo que la red vio al
aprender y lo que ve al evaluarla. Para que el modelo tolere los cambios de luz prefiero usar aumento de datos de
brillo y contraste durante el entrenamiento (Notebook 02).

In [ ]:
# ============================================================
# 6. Efecto de CLAHE sobre los rostros más oscuros
# ============================================================
brillos = np.array([pf.metricas_calidad(x)["brillo"] for x in X_prop])
idx = np.argsort(brillos)[:min(3, len(X_prop))]
fig, axes = plt.subplots(2, len(idx), figsize=(3 * len(idx), 6), squeeze=False)
for j, k in enumerate(idx):
    axes[0, j].imshow(X_prop[k]); axes[0, j].set_title(f"Original (brillo {brillos[k]:.0f})", fontsize=9)
    eq = pf.ecualizar_clahe(X_prop[k])
    axes[1, j].imshow(eq); axes[1, j].set_title(f"CLAHE (brillo {pf.metricas_calidad(eq)['brillo']:.0f})", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout(); plt.savefig(f"{RES}/01_clahe.png", dpi=130); plt.show()

In [ ]:
# ============================================================
# 7. Resumen del dataset para el informe
# ============================================================
resumen = {
    "personas": nombres_prop,
    "imagenes_originales": int(len(registro)),
    "rostros_aceptados": int(registro["aceptada"].sum()),
    "por_persona": {n: int((y_prop == i).sum()) for i, n in enumerate(nombres_prop)},
    "formatos": registro["formato"].value_counts().to_dict(),
    "detectadas_con_reescalado": int((registro.get("escala", pd.Series(dtype=float)).fillna(1) != 1).sum()),
    "angulo_abs_medio": float(registro["angulo"].abs().mean()),
    "nitidez_media": float(registro["nitidez"].mean()),
    "tam_recorte": CONFIG["TAM"],
}
with open(f"{RES}/01_resumen_dataset.json", "w", encoding="utf-8") as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False)
print(json.dumps(resumen, indent=2, ensure_ascii=False))